# Statim: synthetic data on an A100 with vLLM (fast setup)

**Runtime → Change runtime type → A100 GPU → Save.** Then run the cells one after another.

| Cell | What happens | Time |
|---|---|---|
| Setup | GPU check, vLLM, Statim code, exact model revisions | ~8 min |
| 1. Write | **Qwen3-30B-A3B-Instruct-2507** (Apache-2.0) writes candidates, live progress | depends on amount |
| 2. Check | **Phi-4** (MIT, another model family) checks every candidate blind, live progress, then a table and samples | depends on amount |
| 3. Save | copy to Google Drive | 1 min |

One model at a time gets the whole GPU (vLLM batches hundreds of requests). Seed texts: BillSum (CC0),
GovReport (CC-BY-4.0), EUR-Lex-Sum (CC-BY-4.0). Colab has no evaluation texts: the output is marked
*unchecked* and is filtered on pop-os (`tools/synth/leakage.py --filter`) before any training use.


In [ ]:
# ===== SETUP (~8 min) =====
import os, subprocess, sys, time, json, urllib.request
PER_CAPABILITY = 500          # kept items per capability (urgency, nli); raise once quality is confirmed
TASKS = "urgency,nli"
GENERATOR = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"   # FP8 weights run on the A100 via Marlin kernels
VERIFIER = "microsoft/phi-4"
VLLM_VERSION = "0.30.0"

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("FAILED: " + cmd)
    return r.stdout

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     text=True, capture_output=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Runtime > Change runtime type > A100 GPU.")
print("GPU:", gpu)
if "A100" not in gpu and "H100" not in gpu and "L4" not in gpu:
    print("WARNING: this notebook is sized for an A100 (40 GB); on a smaller GPU phi-4 may not fit.")

sh(f"pip -q install vllm=={VLLM_VERSION} huggingface_hub datasets pyarrow")
print(sh("python -c 'import vllm, torch; print(\"vllm\", vllm.__version__, \"torch\", torch.__version__)'"))

os.chdir("/content")
sh("rm -rf /content/statim && git clone -q --depth 1 -b data/synth-v2 https://github.com/BEKO2210/statim /content/statim")
os.chdir("/content/statim")
print(sh("git log --oneline -1"))

from huggingface_hub import HfApi
REV = {m: HfApi().model_info(m).sha for m in (GENERATOR, VERIFIER)}   # exact revisions for provenance
print("revisions:", REV)
print("SETUP OK")

In [ ]:
# ===== 1. WRITE: Qwen3-30B-A3B writes candidates (live progress) =====

def start_vllm(model, revision, extra=""):
    """Start vLLM for ONE model and wait until it answers; prints the log tail while loading."""
    log = open(f"/tmp/vllm-{model.split('/')[-1]}.log", "w")
    proc = subprocess.Popen(f"vllm serve {model} --revision {revision} --port 8000 --max-model-len 4096 "
                            f"--gpu-memory-utilization 0.90 --enable-prefix-caching {extra}",
                            shell=True, stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5).read()
            print(f"vLLM ready with {model} after {time.time() - t0:.0f} s", flush=True)
            return proc
        except Exception:
            if proc.poll() is not None:
                print(open(log.name).read()[-4000:])
                raise SystemExit("vLLM stopped while loading " + model)
            if int(time.time() - t0) % 30 == 0:
                print(f"  loading {model} ... {time.time() - t0:.0f} s", flush=True)
            time.sleep(1)

def stop_vllm(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(5)
    print(sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader"))

try:
    gen = start_vllm(GENERATOR, REV[GENERATOR])
except SystemExit as exc:
    # FP8 MoE kernels are new on Ampere; the official GPTQ-Int4 build (Apache-2.0) is the fallback
    print(exc, "-> falling back to Qwen/Qwen3-30B-A3B-GPTQ-Int4")
    GENERATOR = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"
    REV[GENERATOR] = HfApi().model_info(GENERATOR).sha
    gen = start_vllm(GENERATOR, REV[GENERATOR])
!cd /content/statim && TMPDIR=/tmp python -u tools/synth/grounded.py --stage generate --backend openai --host http://127.0.0.1:8000 --generator-model {GENERATOR} --generator-revision {REV[GENERATOR]} --per-capability {PER_CAPABILITY} --tasks {TASKS} --concurrency 64 --oversample 2.5 --max-jobs-factor 40 --cache /tmp/statim-synth-corpora --candidates /content/candidates.jsonl.gz
stop_vllm(gen)

In [ ]:
# ===== 2. CHECK: Phi-4 checks every candidate blind (live progress), then a table and samples =====
ver = start_vllm(VERIFIER, REV[VERIFIER])
!cd /content/statim && TMPDIR=/tmp python -u tools/synth/grounded.py --stage verify --backend openai --host http://127.0.0.1:8000 --verifier-model {VERIFIER} --verifier-revision {REV[VERIFIER]} --per-capability {PER_CAPABILITY} --tasks {TASKS} --concurrency 64 --candidates /content/candidates.jsonl.gz --out-dir /content/synth-a100 --defer-leakage-check --cache /tmp/statim-synth-corpora --resume
stop_vllm(ver)

import pandas as pd
from IPython.display import display, Markdown
m = json.load(open("/content/synth-a100/manifest.json"))
rows = []
for task, st in m["stats"].items():
    mt = st["metrics"]
    rows.append({"capability": task, "kept": sum(st["counts"].values()), "candidates checked": mt.get("verify_called", 0),
                 "verifier agreement": "%.0f%%" % (100 * (st["verify_acceptance_rate"] or 0)),
                 "rejected: disagree": mt.get("verify_disagree", 0), "rejected: error": mt.get("verify_error", 0)})
display(pd.DataFrame(rows))
print("models:", m["models"])
display(Markdown(open("/content/synth-a100/samples.md").read()[:15000]))

In [ ]:
# ===== 3. SAVE to Google Drive (folder statim-synth) =====
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/statim-synth && cp -r /content/synth-a100 /content/candidates.jsonl.gz* /content/drive/MyDrive/statim-synth/
!ls -la /content/drive/MyDrive/statim-synth